In [13]:
import pandas as pd
import datetime as dt
from acled import Acled
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import average_precision_score, classification_report, precision_recall_curve, confusion_matrix

from test import auprc_score

NameError: name 'pd' is not defined

In [3]:
acled = Acled()

INFO:acled:Access token correctly retrieved.


Train: Data from January 2018 to December 2022. This includes the 2018 Sudanese revolution but excludes the 2023 Civil War.
Test onset civil war: Data from January 2023 to December 2023 which includes the escalation of the civil war.
Test active civil war: Data from January 2024 to December 2025 which includes fluctuations in ongoing civil war.


In [4]:
countries = ["Sudan"]
start_date = "2017-07-01" # TODO validation for 6 month warm up period
end_date = "2024-12-31"

train_start_date = "2018-01-01"
train_end_date = "2022-12-31"

onset_start_date = "2023-01-01"
onset_end_date = "2023-12-31"

active_start_date = "2024-01-01"
active_end_date = "2024-12-31"

In [5]:
all_data = acled.get_data(countries, start_date, end_date)

INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:Requesting data...
INFO:acled:All data successfully fetched.


In [6]:
def mark_conflict_events(df: pd.DataFrame) -> pd.DataFrame:
    # 1 = Conflict event (Y)
    # 0 = Non-conflict (used for features)

    acled_subevent_mapping = {
        # BATTLES (Conflict)
        "Armed clash": 1,
        "Government regains territory": 1,
        "Non-state actor overtakes territory": 1,
        # EXPLOSIONS / REMOTE VIOLENCE (Conflict)
        "Air/drone strike": 1,
        "Chemical weapon": 1,
        "Remote explosive/landmine/IED": 1,
        "Shelling/artillery/missile attack": 1,
        "Suicide bomb": 1,
        "Grenade": 1,
        # VIOLENCE AGAINST CIVILIANS (Conflict)
        "Abduction/forced disappearance": 1,
        "Attack": 1,
        "Sexual violence": 1,
        # RIOTS (Conflict)
        "Mob violence": 1,
        "Violent demonstration": 1,
        # PROTESTS (Non-conflict)
        "Excessive force against protesters": 0,
        "Peaceful protest": 0,
        "Protest with intervention": 0,
        # STRATEGIC DEVELOPMENTS (Non-conflict)
        "Agreement": 0,
        "Arrests": 0,
        "Change to group/activity": 0,
        "Disrupted weapons use": 0,
        "Headquarters or base established": 0,
        "Looting/property destruction": 0,
        "Non-violent transfer of territory": 0,
        "Other": 0,
    }
    df["conflict"] = df["sub_event_type"].apply(lambda x: acled_subevent_mapping[x])
    return df #TODO add validation

In [31]:
def create_regional_monthly_baseline(df):
    df = df.copy()
    df_grouped = (
        df.groupby(["admin2", "year_month"])["conflict"]
        .sum()
        .reset_index(name="conflict_event_count")
    )

    # Build full dataset of all regions and months
    all_regions = df["admin2"].unique()
    all_months = pd.period_range(
        df["year_month"].min(), df["year_month"].max(), freq="M"
    )
    full_index = pd.MultiIndex.from_product(
        [all_regions, all_months], names=["admin2", "year_month"]
    )

    df_grouped = (
        df_grouped.set_index(["admin2", "year_month"])
        .reindex(full_index, fill_value=0)
        .reset_index()
        .sort_values(["admin2", "year_month"])
    )

    df_grouped = df_grouped.sort_values(by=["admin2", "year_month"])

    # Calculate rolling statistics ending at the previous month (t-1)
    df_grouped["rolling_mean_6m"] = df_grouped.groupby("admin2")[
        "conflict_event_count"
    ].transform(lambda x: x.rolling(window=6, min_periods=6).mean().shift(1))

    df_grouped["rolling_std_6m"] = df_grouped.groupby("admin2")[
        "conflict_event_count"
    ].transform(lambda x: x.rolling(window=6, min_periods=6).std().shift(1))

    k = 0.5
    df_grouped["escalation_threshold"] = df_grouped["rolling_mean_6m"] + (
        k * df_grouped["rolling_std_6m"]
    )

    # Define the binary target variable (Is current conflict > historical threshold?)
    df_grouped["target_escalation"] = np.where(
        df_grouped["conflict_event_count"] > df_grouped["escalation_threshold"], 1, 0
    )

    return df_grouped

In [32]:
def pre_process_data(df):
    df = mark_conflict_events(df)

    pivot_df = pd.pivot_table(
        df,
        values="event_id_cnty",
        index=["admin2", "year_month"],
        columns=["sub_event_type"],
        aggfunc="count",
        fill_value=0,
    ).reset_index()

    pivot_df.columns = (
        pivot_df.columns.str.lower()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
    )

    baseline_df = create_regional_monthly_baseline(df)

    fatalities_df = (
        df.groupby(["admin2", "year_month"])["fatalities"].sum().reset_index()
    )

    combined_df = pd.merge(
        baseline_df, pivot_df, on=["admin2", "year_month"], how="left")
    combined_df = pd.merge(
        combined_df, fatalities_df, on=["admin2", "year_month"], how="left")

    event_cols = pivot_df.columns.drop(["admin2", "year_month"]).tolist()
    combined_df[event_cols] = combined_df[event_cols].fillna(0)
    combined_df["fatalities"] = combined_df["fatalities"].fillna(0)

    current_event_cols = event_cols + ["fatalities"]
    lagged_event_cols = ["rolling_mean_6m", "rolling_std_6m", "escalation_threshold"]

    combined_df[current_event_cols] = combined_df[current_event_cols].fillna(0)
    combined_df[current_event_cols] = combined_df.groupby("admin2")[current_event_cols].shift(1)

    predictor_cols = current_event_cols + lagged_event_cols
    combined_df[predictor_cols] = combined_df[predictor_cols].fillna(0)

    combined_df = combined_df.rename(columns={"admin2": "region"})
    return combined_df, predictor_cols

In [33]:
def calculate_conflict_ratio(df):
    count_0 = (df["target_escalation"] == 0).sum()
    count_1 = (df["target_escalation"] == 1).sum()
    ratio = count_0 / count_1

    return {"non-escalation": count_0, "escalation": count_1, "ratio": ratio}

In [34]:
def split_data(df, predictor_cols, target_col, start_date, end_date):
    split_df = df[
        (df["year_month"] >= start_date)
        & (df["year_month"] <= end_date)
    ].copy()

    y = split_df[target_col].copy()
    X = split_df[predictor_cols].copy()

    return split_df, y, X

In [35]:
processed_df, predictor_cols = pre_process_data(all_data)

train_df, y_train, X_train = split_data(processed_df, predictor_cols, "target_escalation", train_start_date, train_end_date)
onset_df, y_onset, X_onset = split_data(processed_df, predictor_cols, "target_escalation", onset_start_date, onset_end_date)
active_df, y_active, X_active = split_data(processed_df, predictor_cols, "target_escalation", active_start_date, active_end_date)

ratios = calculate_conflict_ratio(train_df)


In [36]:
ratios

{'non-escalation': np.int64(9685),
 'escalation': np.int64(1055),
 'ratio': np.float64(9.180094786729859)}

In [38]:
scale_weight = ratios["non-escalation"] / ratios["escalation"]
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=scale_weight,
    eval_metric="aucpr",  # As decided in proposal
    random_state=7,
)

timeseries_cv = TimeSeriesSplit(n_splits=5)

param_grid = { # Gemini prompt
    "max_depth": [5, 7, 9],
    "learning_rate": [0.03, 0.05, 0.1],
    "n_estimators": [300, 600, 900],
    "subsample": [0.7, 0.8],
    "colsample_bytree": [0.7, 0.8],
    "min_child_weight": [1, 3]
}

grid_search = GridSearchCV(
    estimator=xgb_model,
    param_grid=param_grid,
    cv=timeseries_cv,
    scoring="average_precision",
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

# tune threshold on onset (validation)
y_pred_proba_onset = best_model.predict_proba(X_onset)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_onset, y_pred_proba_onset)
f1_scores = (2 * precisions * recalls / (precisions + recalls + 1e-10))[:-1]
optimal_threshold = thresholds[np.argmax(f1_scores)]

# evaluate on onset (test)
y_pred_proba_onset = best_model.predict_proba(X_active)[:, 1]
y_pred_custom_onset = (y_pred_proba_onset >= optimal_threshold).astype(int)

# evaluate active (test)
y_pred_proba_active = best_model.predict_proba(X_active)[:, 1]
y_pred_custom_active = (y_pred_proba_active >= optimal_threshold).astype(int)

print(f"Optimal Parameters: {grid_search.best_params_}")
print(f"Optimal Threshold (from onset): {optimal_threshold:.4f}")
print(f"Onset AUPRC: {average_precision_score(y_onset, y_pred_proba_onset):.4f}")
print(classification_report(y_onset, y_pred_custom_onset))
print(f"Active AUPRC: {average_precision_score(y_active, y_pred_proba_active):.4f}")
print(classification_report(y_active, y_pred_custom_active))


Optimal Parameters: {'colsample_bytree': 0.7, 'learning_rate': 0.03, 'max_depth': 5, 'min_child_weight': 3, 'n_estimators': 300, 'subsample': 0.8}
Optimal Threshold (from onset): 0.5123
Onset AUPRC: 0.1860
              precision    recall  f1-score   support

           0       0.85      0.67      0.75      1782
           1       0.20      0.41      0.27       366

    accuracy                           0.62      2148
   macro avg       0.52      0.54      0.51      2148
weighted avg       0.74      0.62      0.66      2148

Active AUPRC: 0.2902
              precision    recall  f1-score   support

           0       0.86      0.70      0.77      1735
           1       0.29      0.52      0.37       413

    accuracy                           0.66      2148
   macro avg       0.57      0.61      0.57      2148
weighted avg       0.75      0.66      0.69      2148

